# 01 · Load the data

**Project:** _title_ · **Team:** _names_

Downloads the raw data once, checks it, and caches it as Parquet in `data/cache/` (git-ignored). Re-run it only when you want fresh data, and say so in the README.

In [2]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv

DATA = Path("../data")
CACHE = DATA / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
load_dotenv(dotenv_path=find_dotenv())                                   # tokens live in ../.env, never in the notebook

True

## Metadata

Name, publisher, licence and last update of each dataset (see `03_open_data_apis` in the course repository).

In [3]:
DATASET = "mrus-ixpq"   # Socrata id of your dataset https://analisi.transparenciacatalunya.cat/api/v3/views/mrus-ixpq/query.json
meta = requests.get(f"https://analisi.transparenciacatalunya.cat/api/views/{DATASET}.json", timeout=30).json()
meta["name"], meta.get("attribution"), meta.get("license")

('Publicacions de recerca de Catalunya',
 'Departament de Recerca i Universitats',
 {'name': 'See Terms of Use'})

In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from sodapy import Socrata
from tqdm import tqdm

CACHE_DIR = Path("../data/cache")
MAX_WORKERS = 4
MAX_RETRIES = 3
RETRY_BACKOFF = 5  # segundos base (5, 10, 20...)

def fetch_papers_from_year(client, year):
    """Descarga y cachea los papers de un año (síncrono)."""
    data = client.get_all(
        DATASET,
        select='date, doi, ruct, orcidResearchers, orcidDirectors, orcidUncheckedAuthors, orcidUncheckedDirectors',
        where=f'''doi IS NOT NULL 
            AND issn IS NOT NULL 
            AND orcidResearchers IS NOT NULL 
            AND ruct IS NOT NULL 
            AND date BETWEEN "{year}-01-01T00:00:00" AND "{year+1}-01-01T00:00:00" ''',
    )
    df = pd.DataFrame(data)
    if len(df) > 0:
        df = df.sort_values(by='date', ignore_index=True)
        df.to_parquet(path=f'../data/cache/papers_{year}.parquet')
    return year, len(df)


def fetch_with_retry(client, year):
    """Reintenta con backoff exponencial ante 429 / errores transitorios."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fetch_papers_from_year(client, year)
        except Exception as e:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(RETRY_BACKOFF * (2 ** (attempt - 1)))


def main():
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    years = list(range(1949, 2027))

    pending = [y for y in years if not (CACHE_DIR / f"papers_{y}.parquet").exists()]
    skipped = len(years) - len(pending)
    if skipped:
        print(f"{skipped} años ya en caché, {len(pending)} por descargar.")

    if not pending:
        return

    with Socrata(
        domain="analisi.transparenciacatalunya.cat",
        app_token=os.environ['SOCRATA_APP_TOKEN'],
        username=os.environ['SOCRATA_USER_ID'],
        password=os.environ['SOCRATA_USER_PASSWD'],
        timeout=600,
    ) as client, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:

        futures = {ex.submit(fetch_with_retry, client, y): y for y in pending}

        errors = []
        with tqdm(total=len(futures), desc="Descargando papers", unit="año") as pbar:
            for fut in as_completed(futures):
                year = futures[fut]
                try:
                    _, n = fut.result()
                    pbar.set_postfix_str(f"{year}: {n} filas")
                except Exception as e:
                    errors.append((year, e))
                    pbar.set_postfix_str(f"{year}: ERROR")
                pbar.update(1)

    if errors:
        print("\nAños con error:")
        for year, err in errors:
            print(f"  {year}: {type(err).__name__}: {err}")


if __name__ == "__main__":
    main()

66 años ya en caché, 12 por descargar.


Descargando papers: 100%|██████████| 12/12 [1:22:56<00:00, 414.73s/año, 2021: 594481 filas]


In [12]:
all_papers = pd.DataFrame()
for f in tqdm(list(CACHE_DIR.glob("*.parquet"))):
    if (all_papers.empty):
        all_papers = pd.read_parquet(f)
    else:
        all_papers = pd.concat([all_papers, pd.read_parquet(f)], ignore_index=True)

100%|██████████| 77/77 [00:01<00:00, 55.75it/s] 


In [16]:
all_papers

,date,doi,ruct,orcidResearchers,orcidUncheckedAuthors,orcidDirectors
0,1949-01-01T00:00:00.000,10.1210/jcem-9-9-883,Universitat Rovira i Virgili,0000-0002-6245-7933,NaN,NaN
1,1949-01-01T00:00:00.000,10.1210/jcem-9-9-883,Universitat Rovira i Virgili,0000-0002-6245-7933,NaN,NaN
2,1949-01-01T00:00:00.000,10.1210/jcem-9-9-883,Universitat Rovira i Virgili,0000-0002-6245-7933,NaN,NaN
3,1949-01-01T00:00:00.000,10.1210/jcem-9-9-883,Universitat Rovira i Virgili,0000-0002-6245-7933,NaN,NaN
4,1949-01-01T00:00:00.000,10.1210/jcem-9-9-883,Universitat Rovira i Virgili,0000-0002-6245-7933,NaN,NaN
...,...,...,...,...,...,...
8557652,2022-01-01T00:00:00.000,10.3390/sym14101993,Universitat de Vic - Universitat Central de Ca...,0000-0003-1692-311X,NaN,NaN
8557653,2022-01-01T00:00:00.000,10.1212/wnl.0000000000013278,"Institut de Recerca Germans Trias i Pujol,Univ...",0000-0002-3422-9685;0000-0002-1896-1622;0000-0...,NaN,NaN
8557654,2022-01-01T00:00:00.000,10.1177/20539517221082027,Universitat Ramon Llull,0000-0003-2297-3308,NaN,NaN
8557655,2022-01-01T00:00:00.000,10.1344/bid2022.49.14,Universitat Pompeu Fabra,0000-0001-8304-8082,NaN,NaN


In [23]:
cleaned_papers = all_papers.drop_duplicates(subset=['doi'], ignore_index=True)
cleaned_papers.to_parquet('../data/papers.gencat.parquet')

In [30]:
valores_unicos = (
    cleaned_papers['orcidResearchers']
    .str.split(';')
    .explode()
    .drop_duplicates()
    .reset_index(drop=True)
)

valores_unicos

0        0000-0002-6245-7933
1        0000-0001-6112-5913
2        0000-0001-5050-0167
3        0000-0002-6987-8810
4        0000-0001-8124-6210
                ...         
29453    0000-0003-2849-1801
29454    0000-0002-4178-8266
29455    0000-0002-3517-6116
29456    0000-0001-7169-8002
29457    0000-0002-6947-0460
Name: orcidResearchers, Length: 29458, dtype: str

## Download (all rows) and cache

In [ ]:
def cached(name, fetch):
    """Return data/cache/<name>.parquet; download it with fetch() the first time."""
    path = CACHE / f"{name}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    df = fetch()
    df.to_parquet(path, index=False)
    return df




# df = cached("raw", lambda: fetch_socrata(DATASET))   # fetch_socrata: see 03_open_data_apis
# print(len(df), "rows")

## Checks

Row count against the server, types, missing values.

In [ ]:
def profile(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (100 * df.isna().mean()).round(1),
        "n_unique": df.nunique(),
    })


# profile(df)

## Dataset card

| Field | Answer |
|---|---|
| Source (URL) and licence | |
| Rows × columns | |
| What is one row? | |
| Columns we use | |
| Missing values / problems | |
| API access? | |